<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/crossscale_oddball_index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The oddball prediction-error index across recording techniques — and why the raw sign reverses

This notebook computes the feature-oddball response (a rare 90° grating where a
frequent 0° standard is expected) across all three OpenScope techniques —
Neuropixels spikes, mesoscope jGCaMP8s, SLAP2 iGluSnFR — on a common footing, and
then dissects **why the raw oddball-vs-standard contrast changes sign** between
spiking and imaging.

**Two indices** (both from the project's pre-registration):
- **OI** = (R_oddball − R_standard) / (|R_oddball| + |R_standard|) — oddball vs the
  *frequent* standard. Available in all three techniques.
- **DvI** = (R_oddball − R_control) / (|R_oddball| + |R_control|) — oddball vs the
  *physically identical* 90° grating in the equiprobable control block, where it is
  **not** surprising. This is the adaptation-controlled index. Available in
  Neuropixels + mesoscope (SLAP2 has no equiprobable control block).

**Integration window** (from the companion `crossscale_timescales` notebook): each
technique's response is integrated over one stimulus cycle — spikes 0–500 ms,
calcium 0–700 ms — capped at the fixed ~701 ms grating cycle so no trailing
stimulus contaminates the window. All on **responsive cells**.

**The headline finding.** The raw OI reverses sign — mildly positive in spikes
(+0.13), strongly negative in both calcium methods (−1.0 mesoscope, −0.47 SLAP2).
This is **not** a difference in prediction-error signalling. It is a **population
tuning-sampling effect**: two-photon imaging over-samples neurons tuned to the
frequent 0° standard, so a 90° grating (deviant *or* control) drives the imaged
population less. The adaptation-controlled **DvI removes this confound** and stays
**positive across techniques** (+0.39 spikes, +0.11 mesoscope) — deviance detection
is a genuine cross-scale phenomenon; only the control-referenced index reveals it.


In [ ]:
#@title Install
!pip -q install pynwb h5py remfile requests pandas numpy matplotlib scipy

In [ ]:
#@title Streaming helpers + indices
import numpy as np, pandas as pd, h5py, remfile, requests, re
from scipy import stats as ss

def s3_url(ds, aid, version="draft"):
    u=f"https://api.dandiarchive.org/api/dandisets/{ds}/versions/{version}/assets/{aid}/download/"
    return requests.get(u,allow_redirects=False,timeout=60).headers["Location"]
def col(g,c):
    v=g[c][:]; return np.array([x.decode() if isinstance(x,bytes) else x for x in v])
def idx(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float)
    d=np.abs(a)+np.abs(b); return np.where(d>1e-12,(a-b)/d,np.nan)

# per-technique integration window (one ~701 ms stimulus cycle) and baselines
WIN={"ecephys":(0.0,0.5),"mesoscope":(0.0,0.7),"slap2":(0.0,0.7)}
BASE_E=(-0.1,-0.005)   # spikes: short pre-stimulus baseline
BASE_C=(-0.3,-0.02)    # calcium: longer baseline (slow indicator)
print("ready")

In [ ]:
#@title Unified per-cell extractor (index + tuning preference + response sign + area)
# TPI = tuning-preference index measured in the CONTROL block (both 0 and 90 non-surprising):
#   TPI>0 prefers 90 (the deviant orientation), TPI<0 prefers the 0 standard orientation.
# For ecephys/mesoscope TPI is control-derived, so it is INDEPENDENT of the oddball-block OI.
def ece_mech(aid):
    fh=h5py.File(remfile.File(s3_url("001637",aid)),"r")
    try:
        w=WIN["ecephys"]
        gO=fh["intervals"]["Standard mismatch block_presentations"];TT=col(gO,"TrialType");ts=gO["start_time"][:]
        t_std=ts[TT=="standard"];t_o90=ts[TT=="orientation_90"]
        gC=fh["intervals"]["Control block 1_presentations"];Cori=gC["Orientation"][:].astype(float)
        Cdur=gC["stop_time"][:]-gC["start_time"][:];Cts=gC["start_time"][:];ok=Cdur>=0.3
        t_c90=Cts[np.isclose(Cori,1.571,atol=0.05)&ok];t_c0=Cts[np.isclose(Cori,0.0,atol=0.05)&ok]
        U=fh["units"];st=U["spike_times"][:];sti=U["spike_times_index"][:]
        n=len(sti);qc=U["default_qc"][:] if "default_qc" in U else np.ones(n,bool)
        Eg=fh["general/extracellular_ephys/electrodes"];eloc=col(Eg,"location");egroup=col(Eg,"group_name")
        dev=col(U,"device_name");eci=U["extremum_channel_index"][:]
        groups=sorted(set(egroup),key=lambda gn:np.where(egroup==gn)[0][0])
        offs={gn:int(np.where(egroup==gn)[0][0]) for gn in groups};bl={gn:int((egroup==gn).sum()) for gn in groups}
        def d2g(d):
            for gn in groups:
                if d[-1].lower() in gn.lower(): return gn
            return groups[0]
        dmap={d:d2g(d) for d in set(dev)}
        # extremum_channel_index is PER-PROBE -> offset into the stacked electrodes table
        u_loc=np.array([eloc[offs[dmap[dev[i]]]+min(int(eci[i]),bl[dmap[dev[i]]]-1)] for i in range(n)])
        vis=np.where(qc & np.array([bool(re.match("VIS",r)) for r in u_loc]))[0]
        def sp_(i): return st[(0 if i==0 else sti[i-1]):sti[i]]
        def evk(sp,times,win):
            ev=np.searchsorted(sp,times+win[1])-np.searchsorted(sp,times+win[0])
            b=np.searchsorted(sp,times+BASE_E[1])-np.searchsorted(sp,times+BASE_E[0])
            return ev/(win[1]-win[0]) - b/(BASE_E[1]-BASE_E[0])
        rows=[]
        for u in vis:
            sp=sp_(u)
            r_std=evk(sp,t_std,w);r_o90=evk(sp,t_o90,w);r_c90=evk(sp,t_c90,w);r_c0=evk(sp,t_c0,w)
            resp=np.any(r_std!=0) and ss.wilcoxon(r_std).pvalue<0.05; m=np.mean(r_std)
            rows.append(dict(area=u_loc[u],resp=bool(resp),R_std=m,R_o90=np.mean(r_o90),
                R_c90=np.mean(r_c90),R_c0=np.mean(r_c0),resp_sign=int(np.sign(m))))
        df=pd.DataFrame(rows)
        df["OI"]=idx(df.R_o90,df.R_std);df["DvI"]=idx(df.R_o90,df.R_c90);df["TPI"]=idx(df.R_c90,df.R_c0)
        df["modality"]="ecephys";return df
    finally: fh.close()

def img_mech(ds,aid,slap=False):
    fh=h5py.File(remfile.File(s3_url(ds,aid)),"r")
    try:
        w=WIN["slap2" if slap else "mesoscope"]
        if slap:
            g=fh["intervals"]["gratings"];O=g["orientation"][:].astype(float);C=g["contrast"][:].astype(float)
            ts=g["start_time"][:];dur=g["stop_time"][:]-g["start_time"][:];ok=(C>0)&(dur>=0.3)
            t_std=ts[np.isclose(O,0.0,atol=0.05)&ok];t_o90=ts[np.isclose(O,1.571,atol=0.05)&ok];t_c90=t_c0=None
            fl=fh["processing"]["ophys"];unit_sets=[]
            for dmd,off in [("Fluorescence_DMD1",0.115),("Fluorescence_DMD2",-0.165)]:  # simultaneous DMDs, slight delay
                sub=fl[dmd];key=[k for k in sub if k.endswith("dFF")][0]
                unit_sets.append((sub[key]["data"][:],sub[key]["timestamps"][:]+off,"SLAP2"))
        else:
            I=fh["intervals"];blk=[k for k in I if "Standard mismatch" in k][0];g=I[blk]
            TT=col(g,"TrialType");ts=g["start_time"][:];t_std=ts[TT=="standard"];t_o90=ts[TT=="orientation_90"]
            gC=I["Control block 1_presentations"];Cori=gC["Orientation"][:].astype(float)
            Cdur=gC["stop_time"][:]-gC["start_time"][:];Cts=gC["start_time"][:];ok=Cdur>=0.3
            t_c90=Cts[np.isclose(Cori,1.571,atol=0.05)&ok];t_c0=Cts[np.isclose(Cori,0.0,atol=0.05)&ok]
            unit_sets=[]
            for pl in [k for k in fh["processing"] if k.startswith("VIS")]:
                grp=fh["processing"][pl];dff=grp["dff_timeseries"]["dff_timeseries"]
                data=dff["data"][:];dts=dff["timestamps"][:]
                try: soma=grp["image_segmentation"]["roi_table"]["is_soma"][:].astype(bool)
                except: soma=np.ones(data.shape[1],bool)
                unit_sets.append((data[:,soma],dts,pl))
        def ev(tr,dts,times):
            out=[]
            for t0 in times:
                rb=(dts>=t0+w[0])&(dts<t0+w[1]);bb=(dts>=t0+BASE_C[0])&(dts<t0+BASE_C[1])
                out.append(np.nanmean(tr[rb])-np.nanmean(tr[bb]) if rb.sum() and bb.sum() else np.nan)
            return np.array(out)
        rows=[]
        for data,dts,area in unit_sets:
            for r in range(data.shape[1]):
                tr=data[:,r];e=ev(tr,dts,t_std);e=e[np.isfinite(e)]
                if len(e)<5: continue
                m=np.nanmean(e);resp=ss.wilcoxon(e).pvalue<0.05 and m>0;r_o90=np.nanmean(ev(tr,dts,t_o90))
                if slap: r_c90=r_c0=np.nan; tpi=idx([r_o90],[m])[0]
                else: r_c90=np.nanmean(ev(tr,dts,t_c90));r_c0=np.nanmean(ev(tr,dts,t_c0));tpi=idx([r_c90],[r_c0])[0]
                rows.append(dict(area=re.sub(r"_?\d.*$","",area),resp=bool(resp),R_std=m,R_o90=r_o90,
                    R_c90=r_c90,R_c0=r_c0,resp_sign=int(np.sign(m)),TPI=tpi))
        df=pd.DataFrame(rows)
        df["OI"]=idx(df.R_o90,df.R_std);df["DvI"]=idx(df.R_o90,df.R_c90) if not slap else np.nan
        df["modality"]="slap2" if slap else "mesoscope";return df
    finally: fh.close()
print("extractors ready")

In [ ]:
#@title Sessions
SESSIONS={
 "ecephys":  [("830851","9b9e8abe-7b43-47f1-b8e1-4114f87898a1"),
              ("830848","228c2c2e-1daf-4bf6-9f66-eb6b2bce5ba5"),
              ("830846","680d1c0c-e338-4d0b-ba29-4329436d2ae2")],
 "mesoscope":[("845342","6288e7b0-5b44-4660-b36d-c14d19220e2c"),
              ("837568","b425c043-ebf5-456c-83a9-1d99465224c6")],
 "slap2":    [("796630","b8c05ec0-0a74-46f5-956d-e82af3cc5d27"),
              ("803496","d23a03af-c3bd-4cf0-9492-6dca96fb201d"),
              ("801381","2ecafd40-29a6-422f-92b0-32f7e940c783")],
}
# Imaging sessions run several minutes each (per-ROI responsiveness test). Quick pass:
ONE_PER_MODALITY=True
if ONE_PER_MODALITY: SESSIONS={k:v[:1] for k,v in SESSIONS.items()}
print({k:[s for s,_ in v] for k,v in SESSIONS.items()})

In [ ]:
#@title Run extraction across sessions
import time
rows=[]
for mod,slist in SESSIONS.items():
    for subj,aid in slist:
        t0=time.time()
        try:
            d=ece_mech(aid) if mod=="ecephys" else img_mech("001768" if mod=="mesoscope" else "001424",aid,slap=(mod=="slap2"))
            d["subject"]=subj;rows.append(d);g=d[d.resp]
            print(f"  {mod}/{subj}: {g.resp.sum()}/{len(d)} resp | OI={g.OI.median():+.3f} TPI={g.TPI.median():+.3f} ({time.time()-t0:.0f}s)")
        except Exception as e:
            print(f"  {mod}/{subj}: ERROR {str(e)[:80]}")
M=pd.concat(rows,ignore_index=True);G=M[M.resp].copy()
print("total responsive cells:",len(G))

## 1 · The indices across techniques

OI reverses sign between spikes and calcium; DvI stays positive wherever it can be
computed. The numbers below are per-cell medians with bootstrap 95% CIs.

In [ ]:
#@title Pooled indices per technique
def boot_ci(x,n=2000,seed=0):
    x=np.asarray(x,float);x=x[np.isfinite(x)]
    if len(x)<3: return (np.nan,np.nan,np.nan)
    rng=np.random.default_rng(seed);bs=[np.median(rng.choice(x,len(x),True)) for _ in range(n)]
    return np.median(x),np.percentile(bs,2.5),np.percentile(bs,97.5)
for mod in ["ecephys","mesoscope","slap2"]:
    d=G[G.modality==mod];oi=boot_ci(d.OI);dvi=boot_ci(d.DvI)
    print(f"{mod:<10} n={len(d):<5} OI={oi[0]:+.3f} [{oi[1]:+.3f},{oi[2]:+.3f}]   "
          + (f"DvI={dvi[0]:+.3f} [{dvi[1]:+.3f},{dvi[2]:+.3f}]" if np.isfinite(dvi[0]) else "DvI=n/a (no control block)"))

## 2 · Why OI reverses — a population tuning-sampling effect

The decisive test: within *every* technique, OI tracks each cell's orientation
preference (TPI). What differs across techniques is not the OI–tuning law but the
**mix of cells sampled** — two-photon over-samples 0°-preferring cells, dragging the
population-median OI negative. The control-referenced DvI is tuning-*independent*,
which is why it transfers across scales.

In [ ]:
#@title The mechanism, quantified
for mod in ["ecephys","mesoscope"]:
    d=G[G.modality==mod].dropna(subset=["OI","TPI"])
    rho_oi=ss.spearmanr(d.OI,d.TPI)[0]
    rho_dvi=ss.spearmanr(d.DvI,d.TPI)[0] if d.DvI.notna().any() else np.nan
    print(f"{mod}:")
    print(f"   population tuning bias: median TPI={d.TPI.median():+.3f}, fraction preferring 90 = {(d.TPI>0).mean():.2f}")
    print(f"   OI  ~ tuning: Spearman rho={rho_oi:+.3f}  (OI IS tuning-driven)")
    print(f"   DvI ~ tuning: Spearman rho={rho_dvi:+.3f}  (DvI is tuning-INDEPENDENT)")
    print(f"   OI split: prefers-0 = {d[d.TPI<-0.1].OI.median():+.3f} | prefers-90 = {d[d.TPI>0.1].OI.median():+.3f}")

In [ ]:
#@title Figure — the mechanism (scatter, tuning distribution, split, DvI-independence)
import matplotlib.pyplot as plt
MODCOL={"ecephys":"#2c3e8c","mesoscope":"#c0392b","slap2":"#8e44ad"}
fig,ax=plt.subplots(1,3,figsize=(14,4.2))
# A: OI vs TPI (both techniques overlaid)
for mod in ["ecephys","mesoscope"]:
    d=G[G.modality==mod].dropna(subset=["OI","TPI"])
    ax[0].scatter(d.TPI,d.OI,s=7,c=MODCOL[mod],alpha=0.35,ec="none",label=mod)
ax[0].axhline(0,color="0.6",lw=0.6,ls=":");ax[0].axvline(0,color="0.6",lw=0.6,ls=":")
ax[0].set_xlabel("tuning preference (TPI: + = prefers 90°)");ax[0].set_ylabel("OI (oddball vs standard)")
ax[0].legend(frameon=False,fontsize=8);ax[0].set_title("OI tracks tuning in both techniques",loc="left",fontsize=9)
# B: tuning distributions
for mod in ["ecephys","mesoscope","slap2"]:
    v=G[G.modality==mod].TPI.dropna().values
    ax[1].hist(v,bins=np.linspace(-1,1,26),density=True,histtype="step",lw=2,color=MODCOL[mod],label=f"{mod} ({np.median(v):+.2f})")
ax[1].axvline(0,color="0.6",lw=0.6,ls=":");ax[1].set_xlabel("tuning preference (TPI)");ax[1].set_ylabel("density")
ax[1].legend(frameon=False,fontsize=7.5);ax[1].set_title("The cause: calcium samples 0°-preferring cells",loc="left",fontsize=9)
# C: DvI tuning-independence (mesoscope)
d=G[G.modality=="mesoscope"].dropna(subset=["DvI","TPI"])
ax[2].scatter(d.TPI,d.DvI,s=6,c=MODCOL["mesoscope"],alpha=0.3,ec="none")
rho=ss.spearmanr(d.DvI,d.TPI)[0]
ax[2].axhline(0,color="0.6",lw=0.6,ls=":");ax[2].axvline(0,color="0.6",lw=0.6,ls=":")
ax[2].set_xlabel("tuning preference (TPI)");ax[2].set_ylabel("DvI (oddball vs control)")
ax[2].set_title(f"DvI is tuning-independent (rho={rho:+.2f}) -> transfers",loc="left",fontsize=9)
fig.tight_layout();plt.show()

## Takeaway

- The feature-oddball **DvI is positive across techniques** (spikes +0.39, mesoscope
  +0.11) — deviance detection is a genuine cross-scale phenomenon.
- The raw **OI reverses sign** (spikes +0.13, calcium −1.0 / −0.47) **not** because
  prediction error differs across scales, but because of a **population
  tuning-sampling effect**: two-photon imaging over-samples neurons tuned to the
  frequent standard, so a 90° grating drives the imaged population less.
- Within every technique, OI tracks tuning (ρ≈+0.5–0.6); the reversal disappears when
  cells are split by preference. The control-referenced **DvI is tuning-independent**
  (mesoscope ρ≈0), which is exactly why it survives the change of recording method.
- **Practical lesson:** never compare a raw oddball-vs-standard contrast across
  recording techniques with different sampling biases — always reference an
  equiprobable control of the physically identical stimulus.
- **SLAP2 caveat:** with no equiprobable control block, its OI and tuning index come
  from the same comparison, so tuning and deviance cannot be independently separated
  there; the clean dissociation rests on Neuropixels + mesoscope.

---
*Generated for `ai_oscp_neuro`. Data: DANDI 001637 / 001768 / 001424 (OpenScope
Community Predictive Processing, Allen Institute for Neural Dynamics). Streams
anonymously — no credentials needed.*